In [60]:
#%pip install ipython-sql prettytable==3.6.0
#%pip install pandas
#%pip install ipython-sql
%load_ext sql
import csv, sqlite3
import pandas as pd
import prettytable
from io import StringIO
prettytable.DEFAULT = 'DEFAULT'


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [61]:
con = sqlite3.connect("FinalaDB.db")
cur = con.cursor()
%sql sqlite:///FinalaDB.db


In [ ]:
census = pd.read_csv('ChicagoCensusData.csv')
census.to_sql("ChicagoCensusData", con, if_exists='replace',
              index=False, method="multi")


78

In [ ]:
cps = pd.read_csv('ChicagoPublicSchools.csv')
cps.to_sql("ChicagoPublicSchools", con,
           if_exists='replace', index=False, method="multi")


566

In [ ]:
ccs = pd.read_csv('ChicagoCrimeData.csv')
ccs.to_sql("ChicagoCrimeData", con, if_exists='replace',
           index=False, method="multi")


533

In [65]:
#Find the total number of crimes recorded in the CRIME table.
%sql select count(*) from ChicagoCrimeData;


 * sqlite:///FinalaDB.db
Done.


count(*)
533


In [66]:
#List community area names and numbers with per capita income less than 11000.
%sql select COMMUNITY_AREA_NAME,COMMUNITY_AREA_NUMBER FROM ChicagoCensusData WHERE PER_CAPITA_INCOME < 11000;


 * sqlite:///FinalaDB.db
Done.


COMMUNITY_AREA_NAME,COMMUNITY_AREA_NUMBER
West Garfield Park,26.0
South Lawndale,30.0
Fuller Park,37.0
Riverdale,54.0


In [67]:
#List all case numbers for crimes involving minors?(children are not considered minors for the purposes of crime analysis)
%sql select CASE_NUMBER, DESCRIPTION from chicagoCrimeData where DESCRIPTION like '%minor%';


 * sqlite:///FinalaDB.db
Done.


CASE_NUMBER,DESCRIPTION
HL266884,SELL/GIVE/DEL LIQUOR TO MINOR
HK238408,ILLEGAL CONSUMPTION BY MINOR


In [68]:
#List all kidnapping crimes involving a child?
%sql select * from chicagoCrimeData where PRIMARY_TYPE = 'KIDNAPPING' AND DESCRIPTION LIKE '%CHILD%';


 * sqlite:///FinalaDB.db
Done.


ID,CASE_NUMBER,DATE,BLOCK,IUCR,PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,BEAT,DISTRICT,WARD,COMMUNITY_AREA_NUMBER,FBICODE,X_COORDINATE,Y_COORDINATE,YEAR,LATITUDE,LONGITUDE,LOCATION
5276766,HN144152,2007-01-26,050XX W VAN BUREN ST,1792,KIDNAPPING,CHILD ABDUCTION/STRANGER,STREET,0,0,1533,15,29.0,25.0,20,1143050.0,1897546.0,2007,41.87490841,-87.75024931,"(41.874908413, -87.750249307)"


In [69]:
#List the kind of crimes that were recorded at schools. (No repetitions
%sql select DISTINCT(PRIMARY_TYPE) from chicagoCrimeData where LOCATION_DESCRIPTION  LIKE '%SCHOOL%';


 * sqlite:///FinalaDB.db
Done.


PRIMARY_TYPE
BATTERY
CRIMINAL DAMAGE
NARCOTICS
ASSAULT
CRIMINAL TRESPASS
PUBLIC PEACE VIOLATION


In [70]:
#List the type of schools along with the average safety score for each type
%sql select `Elementary, Middle, or High School` as schooltype, avg(SAFETY_SCORE) as avgss FROM ChicagoPublicSchools group by schooltype;


 * sqlite:///FinalaDB.db
Done.


schooltype,avgss
ES,49.52038369304557
HS,49.62352941176471
MS,48.0


In [71]:
#List 5 community areas with highest % of households below poverty line
%sql select * from ChicagoCensusData group by PERCENT_HOUSEHOLDS_BELOW_POVERTY ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC LIMIT 5;


 * sqlite:///FinalaDB.db
Done.


COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PERCENT_OF_HOUSING_CROWDED,PERCENT_HOUSEHOLDS_BELOW_POVERTY,PERCENT_AGED_16__UNEMPLOYED,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,PERCENT_AGED_UNDER_18_OR_OVER_64,PER_CAPITA_INCOME,HARDSHIP_INDEX
54.0,Riverdale,5.8,56.5,34.6,27.5,51.5,8201,98.0
37.0,Fuller Park,3.2,51.2,33.9,26.6,44.9,10432,97.0
68.0,Englewood,3.8,46.6,28.0,28.5,42.5,11888,94.0
29.0,North Lawndale,7.4,43.1,21.2,27.6,42.7,12034,87.0
27.0,East Garfield Park,8.2,42.4,19.6,21.3,43.2,12961,83.0


In [72]:
%sql SELECT COMMUNITY_AREA_NAME, PERCENT_HOUSEHOLDS_BELOW_POVERTY \
FROM ChicagoCensusData \
ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC \
LIMIT 5;


 * sqlite:///FinalaDB.db
Done.


COMMUNITY_AREA_NAME,PERCENT_HOUSEHOLDS_BELOW_POVERTY
Riverdale,56.5
Fuller Park,51.2
Englewood,46.6
North Lawndale,43.1
East Garfield Park,42.4


In [73]:
#Which community area is most crime prone? Display the coumminty area number only.
%sql select COMMUNITY_AREA_NUMBER,count(*) as TotalCrimes FROM ChicagoCrimeData group by COMMUNITY_AREA_NUMBER ORDER BY TotalCrimes desc limit 1;


 * sqlite:///FinalaDB.db
Done.


COMMUNITY_AREA_NUMBER,TotalCrimes
25.0,43


In [74]:
%sql Select COMMUNITY_AREA_NAME ,max(HARDSHIP_INDEX)from ChicagoCensusData


 * sqlite:///FinalaDB.db
Done.


COMMUNITY_AREA_NAME,max(HARDSHIP_INDEX)
Riverdale,98.0


In [75]:
#Use a sub-query to find the name of the community area with highest hardship index
%sql select COMMUNITY_AREA_NAME from (Select COMMUNITY_AREA_NAME ,max(HARDSHIP_INDEX)from ChicagoCensusData);


 * sqlite:///FinalaDB.db
Done.


COMMUNITY_AREA_NAME
Riverdale


In [76]:
#Use a sub-query to find the name of the community area with highest hardship index
%sql select COMMUNITY_AREA_NAME from ChicagoCensusData where HARDSHIP_INDEX = (Select max(HARDSHIP_INDEX)from ChicagoCensusData);


 * sqlite:///FinalaDB.db
Done.


COMMUNITY_AREA_NAME
Riverdale


In [77]:
%%sql
WITH Crimes AS (
    SELECT COMMUNITY_AREA_NUMBER, COUNT(*) AS TotalCrimes 
    FROM ChicagoCrimeData 
    GROUP BY COMMUNITY_AREA_NUMBER
)
SELECT COMMUNITY_AREA_NAME, COMMUNITY_AREA_NUMBER 
FROM ChicagoCensusData 
WHERE COMMUNITY_AREA_NUMBER IN (
    SELECT COMMUNITY_AREA_NUMBER FROM Crimes 
    WHERE TotalCrimes = (SELECT MAX(TotalCrimes) FROM Crimes)
);


 * sqlite:///FinalaDB.db
Done.


COMMUNITY_AREA_NAME,COMMUNITY_AREA_NUMBER
Austin,25.0


In [78]:
#Use a sub-query to find the name of the community area with highest hardship index
%sql select COMMUNITY_AREA_NAME from ChicagoCensusData where HARDSHIP_INDEX in (Select max(HARDSHIP_INDEX)from ChicagoCensusData);


 * sqlite:///FinalaDB.db
Done.


COMMUNITY_AREA_NAME
Riverdale


In [79]:
%%sql 
WITH u AS (
    SELECT * 
    FROM ChicagoCrimeData 
    JOIN ChicagoCensusData 
    ON ChicagoCrimeData.COMMUNITY_AREA_NUMBER = ChicagoCensusData.COMMUNITY_AREA_NUMBER
),
ttcrimes AS (
    SELECT COMMUNITY_AREA_NUMBER, COUNT(*) AS TotalCrimes 
    FROM u 
    GROUP BY COMMUNITY_AREA_NUMBER
)
SELECT COMMUNITY_AREA_NUMBER, MAX(TotalCrimes) FROM ttcrimes;
   

 * sqlite:///FinalaDB.db
Done.


COMMUNITY_AREA_NUMBER,MAX(TotalCrimes)
25.0,43


In [80]:
%%sql
CREATE VIEW BASIC AS
SELECT * 
    FROM ChicagoCrimeData 
    JOIN ChicagoCensusData 
    ON ChicagoCrimeData.COMMUNITY_AREA_NUMBER = ChicagoCensusData.COMMUNITY_AREA_NUMBER



 * sqlite:///FinalaDB.db
(sqlite3.OperationalError) view BASIC already exists
[SQL: CREATE VIEW BASIC AS
SELECT * 
    FROM ChicagoCrimeData 
    JOIN ChicagoCensusData 
    ON ChicagoCrimeData.COMMUNITY_AREA_NUMBER = ChicagoCensusData.COMMUNITY_AREA_NUMBER]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [81]:
%%sql
select  * from BASIC limit 5;


 * sqlite:///FinalaDB.db
Done.


ID,CASE_NUMBER,DATE,BLOCK,IUCR,PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,BEAT,DISTRICT,WARD,COMMUNITY_AREA_NUMBER,FBICODE,X_COORDINATE,Y_COORDINATE,YEAR,LATITUDE,LONGITUDE,LOCATION,COMMUNITY_AREA_NUMBER:1,COMMUNITY_AREA_NAME,PERCENT_OF_HOUSING_CROWDED,PERCENT_HOUSEHOLDS_BELOW_POVERTY,PERCENT_AGED_16__UNEMPLOYED,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,PERCENT_AGED_UNDER_18_OR_OVER_64,PER_CAPITA_INCOME,HARDSHIP_INDEX
3512276,HK587712,2004-08-28,047XX S KEDZIE AVE,890,THEFT,FROM BUILDING,SMALL RETAIL STORE,0,0,911,9,14.0,58.0,6,1155838.0,1873050.0,2004,41.8074405,-87.70395585,"(41.8074405, -87.703955849)",58.0,Brighton Park,14.4,23.6,13.9,45.1,39.3,13089,84.0
3406613,HK456306,2004-06-26,009XX N CENTRAL PARK AVE,820,THEFT,$500 AND UNDER,OTHER,0,0,1112,11,27.0,23.0,6,1152206.0,1906127.0,2004,41.89827996,-87.71640551,"(41.898279962, -87.716405505)",23.0,Humboldt park,14.8,33.9,17.3,35.4,38.0,13781,85.0
8002131,HT233595,2011-04-04,043XX S WABASH AVE,820,THEFT,$500 AND UNDER,NURSING HOME/RETIREMENT HOME,0,0,221,2,3.0,38.0,6,1177436.0,1876313.0,2011,41.81593313,-87.62464213,"(41.815933131, -87.624642127)",38.0,Grand Boulevard,3.3,29.3,24.3,15.9,39.5,23472,57.0
7903289,HT133522,2010-12-30,083XX S KINGSTON AVE,840,THEFT,FINANCIAL ID THEFT: OVER $300,RESIDENCE,0,0,423,4,7.0,46.0,6,1194622.0,1850125.0,2010,41.74366532,-87.56246276,"(41.743665322, -87.562462756)",46.0,South Chicago,4.7,29.8,19.7,26.6,41.1,16579,75.0
10402076,HZ138551,2016-02-02,033XX W 66TH ST,820,THEFT,$500 AND UNDER,ALLEY,0,0,831,8,15.0,66.0,6,1155240.0,1860661.0,2016,41.7734553,-87.70648047,"(41.773455295, -87.706480471)",66.0,Chicago Lawn,7.6,27.9,17.1,31.2,40.6,13231,80.0


In [ ]:
"""
#VIEWS 
SELECT  F_NAME,L_NAME,SEX, ADDRESS,JOB_TITLE,DEP_NAME,START_DATE FROM EMPLOYEES JOIN JOBS ON EMPLOYEES.JOB_ID = JOBS.JOB_IDENT JOIN DEPARTMENTS ON EMPLOYEES.DEP_ID = DEPARTMENTS.DEPT_ID_DEP JOIN JOB_HISTORY ON EMPLOYEES.JOB_ID = JOB_HISTORY.JOBS_ID;

SELECT  F_NAME,L_NAME,SEX, ADDRESS,JOB_TITLE,DEP_NAME,START_DATE FROM EMPLOYEES E JOIN JOBS J ON E.JOB_ID = J.JOB_IDENT JOIN DEPARTMENTS D ON E.DEP_ID = D.DEPT_ID_DEP JOIN JOB_HISTORY H ON E.JOB_ID = H.JOBS_ID;

CREATE OR REPLACE VIEW EMP_DEPT AS
SELECT EMP_ID, F_NAME, L_NAME, DEP_ID
FROM EMPLOYEES;

CREATE OR REPLACE VIEW EMPSALARY AS
SELECT EMP_ID, F_NAME, L_NAME, B_DATE, SEX, JOB_TITLE,
MIN_SALARY, MAX_SALARY
FROM EMPLOYEES E JOIN JOBS J ON E.JOB_ID = J.JOB_IDENT;


"""


'\n#VIEWS \nSELECT  F_NAME,L_NAME,SEX, ADDRESS,JOB_TITLE,DEP_NAME,START_DATE FROM EMPLOYEES JOIN JOBS ON EMPLOYEES.JOB_ID = JOBS.JOB_IDENT JOIN DEPARTMENTS ON EMPLOYEES.DEP_ID = DEPARTMENTS.DEPT_ID_DEP JOIN JOB_HISTORY ON EMPLOYEES.JOB_ID = JOB_HISTORY.JOBS_ID;\n\nSELECT  F_NAME,L_NAME,SEX, ADDRESS,JOB_TITLE,DEP_NAME,START_DATE FROM EMPLOYEES E JOIN JOBS J ON E.JOB_ID = J.JOB_IDENT JOIN DEPARTMENTS D ON E.DEP_ID = D.DEPT_ID_DEP JOIN JOB_HISTORY H ON E.JOB_ID = H.JOBS_ID;\n\nCREATE OR REPLACE VIEW EMP_DEPT AS\nSELECT EMP_ID, F_NAME, L_NAME, DEP_ID\nFROM EMPLOYEES;\n\nCREATE OR REPLACE VIEW EMPSALARY AS\nSELECT EMP_ID, F_NAME, L_NAME, B_DATE, SEX, JOB_TITLE,\nMIN_SALARY, MAX_SALARY\nFROM EMPLOYEES E JOIN JOBS J ON E.JOB_ID = J.JOB_IDENT;\n\n\n'

In [ ]:
# stored proceduere statement.

''' 
DELIMITER //

CREATE PROCEDURE RETRIEVE_ALL()

BEGIN
   SELECT *  FROM PETSALE;
END //
DELIMITER ;



DROP PROCEDURE RETRIEVE_ALL;

CALL RETRIEVE_ALL;


'''


' \nDELIMITER //\n\nCREATE PROCEDURE RETRIEVE_ALL()\n\nBEGIN\n   SELECT *  FROM PETSALE;\nEND //\nDELIMITER ;\n\n\n\nDROP PROCEDURE RETRIEVE_ALL;\n\nCALL RETRIEVE_ALL;\n\n\n'

In [ ]:
'''
DELIMITER @
CREATE PROCEDURE UPDATE_SALEPRICE (IN Animal_ID INTEGER, IN Animal_Health VARCHAR(5))
BEGIN
    IF Animal_Health = 'BAD' THEN
        UPDATE PETSALE
        SET SALEPRICE = SALEPRICE - (SALEPRICE * 0.25)
        WHERE ID = Animal_ID;
    ELSEIF Animal_Health = 'WORSE' THEN
        UPDATE PETSALE
        SET SALEPRICE = SALEPRICE - (SALEPRICE * 0.5)
        WHERE ID = Animal_ID;
    ELSE
        UPDATE PETSALE
        SET SALEPRICE = SALEPRICE
        WHERE ID = Animal_ID;
    END IF;
END @

DELIMITER ;

   CALL RETRIEVE_ALL;

   CALL UPDATE_SALEPRICE(1, 'BAD');

   CALL RETRIEVE_ALL;

'''


"\nDELIMITER @\nCREATE PROCEDURE UPDATE_SALEPRICE (IN Animal_ID INTEGER, IN Animal_Health VARCHAR(5))\nBEGIN\n    IF Animal_Health = 'BAD' THEN\n        UPDATE PETSALE\n        SET SALEPRICE = SALEPRICE - (SALEPRICE * 0.25)\n        WHERE ID = Animal_ID;\n    ELSEIF Animal_Health = 'WORSE' THEN\n        UPDATE PETSALE\n        SET SALEPRICE = SALEPRICE - (SALEPRICE * 0.5)\n        WHERE ID = Animal_ID;\n    ELSE\n        UPDATE PETSALE\n        SET SALEPRICE = SALEPRICE\n        WHERE ID = Animal_ID;\n    END IF;\nEND @\n\nDELIMITER ;\n\n\n   CALL RETRIEVE_ALL;\n\n   CALL UPDATE_SALEPRICE(1, 'BAD');\n\n   CALL RETRIEVE_ALL;\n\n\n"

In [ ]:
# A transaction is simply a sequence of operations performed using one or more SQL statements as a single logical unit of work. A database transaction must be ACID (Atomic, Consistent, Isolated and Durable).
